[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/08_Neural_Network_MNIST.ipynb)

# Neural Networks — Handwritten Digit Recognition (MNIST)

**Problem type:** supervised multiclass classification (digits 0–9) with a deep neural network

---

## How a Feed-Forward Neural Network Works

A **feed-forward (or fully-connected) neural network** is organised in *layers*:

| Layer | Role |
|-------|------|
| **Input** | Receives raw pixel values (one node per pixel) |
| **Hidden layers** | Learn increasingly abstract features |
| **Output** | 10 nodes — one probability per digit class |

**Key concepts**

- **Weights & biases** — each connection between neurons has a learnable weight; the network adjusts these during training.
- **Activation functions** — after each linear combination, a non-linear function is applied so the network can model complex patterns.
  - **ReLU** (`max(0, x)`) is used in hidden layers: cheap, fast, avoids vanishing gradients.
  - **Softmax** on the output layer converts raw scores (*logits*) into a proper probability distribution that sums to 1.
- **Loss** — *sparse categorical cross-entropy* measures how far the predicted probabilities are from the true one-hot labels.
- **Backpropagation** — the gradient of the loss with respect to every weight is computed via the chain rule and used to nudge weights in the direction that reduces the loss.
- **Adam optimiser** — an adaptive gradient-descent algorithm that keeps per-parameter learning-rate estimates; generally converges faster than plain SGD.
- **Dropout** — during training, a random fraction of neurons are temporarily zeroed out each forward pass, preventing co-adaptation and reducing overfitting.

**Dataset: MNIST**  
70 000 grayscale 28 × 28 images of handwritten digits (0–9); 60 000 for training, 10 000 for testing. Widely considered the "Hello World" of computer vision.


In [ ]:
# ── 1. Setup & reproducibility ──────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (safe for Colab too)
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version:      {np.__version__}')


In [ ]:
# ── 2. Load MNIST ────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print('Raw shapes')
print(f'  x_train: {x_train.shape}  y_train: {y_train.shape}')
print(f'  x_test : {x_test.shape}   y_test : {y_test.shape}')
print(f'  Pixel range before normalisation: [{x_train.min()}, {x_train.max()}]')

# Normalise pixels to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print(f'  Pixel range after  normalisation: [{x_train.min():.1f}, {x_train.max():.1f}]')

# ── Show sample digits ───────────────────────────────────────────────────
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray_r')
    ax.set_title(f'Label: {y_train[i]}', fontsize=8)
    ax.axis('off')
fig.suptitle('Sample MNIST Training Images', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/_mnist_samples.png', bbox_inches='tight')
plt.show()
print('Grid of 32 sample digits displayed.')


In [ ]:
# ── 3. Exploratory Data Analysis — class balance ─────────────────────────
classes, train_counts = np.unique(y_train, return_counts=True)
_,       test_counts  = np.unique(y_test,  return_counts=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, split, color in zip(
        axes,
        [train_counts, test_counts],
        ['Training set  (60 000 images)', 'Test set  (10 000 images)'],
        ['steelblue', 'darkorange']):
    bars = ax.bar(classes, counts, color=color, edgecolor='white', linewidth=0.8)
    ax.set_xlabel('Digit class', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(split, fontsize=12)
    ax.set_xticks(classes)
    ax.bar_label(bars, padding=3, fontsize=8)
    ax.set_ylim(0, max(counts) * 1.15)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Class Distribution — MNIST is well balanced', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/_mnist_class_balance.png', bbox_inches='tight')
plt.show()
print('Class counts (train):', dict(zip(classes, train_counts)))
print('Class counts (test) :', dict(zip(classes, test_counts)))


In [ ]:
# ── 4. Build the MLP model ───────────────────────────────────────────────
#
# Architecture:
#   Flatten  → converts each 28×28 image into a 784-length vector
#   Dense(256, relu)  → first hidden layer
#   Dropout(0.3)      → randomly zero 30 % of activations during training
#   Dense(128, relu)  → second hidden layer
#   Dropout(0.3)
#   Dense(10, softmax)→ output: probability for each digit 0-9

model = keras.Sequential([
    layers.Input(shape=(28, 28), name='input'),
    layers.Flatten(name='flatten'),
    layers.Dense(256, activation='relu', name='dense_1'),
    layers.Dropout(0.3, name='dropout_1'),
    layers.Dense(128, activation='relu', name='dense_2'),
    layers.Dropout(0.3, name='dropout_2'),
    layers.Dense(10,  activation='softmax', name='output'),
], name='MLP_MNIST')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()


In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────────
# epochs=10 gives ~98 % test accuracy; change to experiment
history = model.fit(
    x_train, y_train,
    validation_split=0.1,   # use 10 % of training data for validation
    epochs=10,
    batch_size=128,
    verbose=1,
)


In [ ]:
# ── 6. Evaluate on held-out test set ─────────────────────────────────────
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'\nTest loss    : {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}  ({test_acc*100:.2f} %)')


In [ ]:
# ── 7. Learning curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, metric, title in zip(
        axes,
        ['accuracy', 'loss'],
        ['Accuracy over epochs', 'Loss over epochs']):
    ax.plot(history.history[metric],         label='train',      linewidth=2)
    ax.plot(history.history[f'val_{metric}'], label='validation', linewidth=2, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric.capitalize())
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Training vs Validation — MLP on MNIST', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/_mnist_learning_curves.png', bbox_inches='tight')
plt.show()
print('Learning curves saved.')


In [ ]:
# ── 8. Confusion matrix ──────────────────────────────────────────────────
y_pred_proba = model.predict(x_test, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=list(range(10)))
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set Predictions', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/_mnist_confusion.png', bbox_inches='tight')
plt.show()

# Per-class accuracy
print('\nPer-class accuracy:')
for digit in range(10):
    mask = y_test == digit
    acc_d = (y_pred[mask] == digit).mean()
    print(f'  Digit {digit}: {acc_d*100:.1f} %')


In [ ]:
# ── 9. Display misclassified digits ─────────────────────────────────────
wrong_idx   = np.where(y_pred != y_test)[0]
show_n      = min(32, len(wrong_idx))
sample_idx  = wrong_idx[:show_n]

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(x_test[idx], cmap='gray_r')
    ax.set_title(f'P:{y_pred[idx]} T:{y_test[idx]}', fontsize=8,
                 color='red' if y_pred[idx] != y_test[idx] else 'green')
    ax.axis('off')
# hide any unused axes
for ax in list(axes.flat)[show_n:]:
    ax.axis('off')

fig.suptitle(f'Misclassified digits  (P = predicted, T = true label)  '
             f'— {len(wrong_idx)} errors out of {len(y_test)}',
             fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/_mnist_misclassified.png', bbox_inches='tight')
plt.show()
print(f'Total misclassifications: {len(wrong_idx)} / {len(y_test)}')


## Findings

| Metric | Value |
|--------|-------|
| **Test accuracy** | ~98 % |
| **Test loss** | ~0.07 |
| **Total parameters** | ~235 k |

**Learning curves**
- Both training and validation accuracy rise steeply in the first 2–3 epochs, then plateau — the model converges quickly on MNIST.
- The training/validation curves remain close throughout, confirming that Dropout (0.3) is effectively suppressing overfitting.
- Loss curves mirror this: steady decrease, no divergence between train and val.

**Confusion matrix / error analysis**
- The most common confusions involve visually similar digit pairs: **4 ↔ 9**, **3 ↔ 5**, and **7 ↔ 2**.
- These errors are understandable even for humans: a carelessly written 4 can look like a 9, and a 3 can resemble a 5.
- A CNN (see *Try it yourself*) can push accuracy above 99.5 % by exploiting local spatial structure.


## Try it yourself

1. **Add/remove layers** — insert another `Dense` layer between the two existing ones; observe how train time and accuracy change.
2. **Remove Dropout** — delete both `Dropout` lines, retrain, and watch the train/val accuracy gap widen as the model overfits.
3. **Switch to a CNN** — replace the `Flatten + Dense` block with Conv2D layers:
   ```python
   layers.Reshape((28, 28, 1)),          # add channel dimension
   layers.Conv2D(32, 3, activation='relu'),
   layers.MaxPooling2D(),
   layers.Conv2D(64, 3, activation='relu'),
   layers.MaxPooling2D(),
   layers.Flatten(),
   layers.Dense(128, activation='relu'),
   ```
   A CNN routinely reaches **99.5 %** test accuracy on MNIST.
4. **Change the optimiser** — replace `'adam'` with `'sgd'` or `keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)`; compare convergence speed.
5. **Tune Dropout** — try 0.1, 0.5; notice the trade-off between regularisation strength and learning capacity.
6. **Increase neurons** — double both hidden-layer sizes (512, 256) and compare parameter count vs. accuracy gain.
